### Sample generation for annual probability rasters
1. Create a binary mask of all the agricultural lands from 2000-2022 (note that the land cover dataset for 2012 is not available).
2. Identify the stable agricultural areas by finding the intersection of all the binary masks.
3. Remove the small pixel clusters, field boundaries using the morphological operations.
4. Convert the final stable agricultural land pixels to points
5. Remove all the points except one inside a 1000/2000 meter buffer ensuring all the points are far from each other (spatial autocorrelation).
6. Final check using a phenology profile of the points to ensure they strictly exhibit agricultural characteristics (sharp increase and decrease in NDVI).
7. Repeat the same for the stable non-agricultural lands. Also I will do a stratified sampling using the 2022 land cover dataset so that the model gets trained will all non-cropland classses and does not get confused between different classes. Importantly, I will ensure that the dominant land cover classes like forest and rangeland, bare soil/sand, water and builtup are represented in the training samples.

In [12]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
import config

# First for stable Ag

### Steps 1-2: Stable ag mask generation

In [13]:
frtc_lc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")
ag_ic = frtc_lc_ic.map(lambda image: image.eq(7).rename("ag"))
stable_ag_mask = ag_ic.sum().eq(22)

### Steps 3: Stable ag mask morphological op

In [14]:
stable_ag_eroded_mask = stable_ag_mask.focalMin(radius=2, units='pixels') #Removing ag field boundary by 2 pixel
patch_size = stable_ag_eroded_mask.connectedPixelCount(maxSize=10, eightConnected=True)
stable_ag_final_mask = stable_ag_eroded_mask.updateMask(patch_size.gte(10))

In [29]:
stable_ag_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

In [30]:
stable_ag_eroded_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

In [31]:
stable_ag_final_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

In [34]:
# Map.addLayer(stable_ag_mask.selfMask(), {"palette":["red"]}, "stable_ag")
# Map.addLayer(stable_ag_eroded_mask.selfMask(), {"palette":["green"]}, "stable_ag_eroded")
# Map.addLayer(stable_ag_final_mask.selfMask(), {"palette":["blue"]}, "stable_ag_final")
# Map

### Step 4-5: Stable ag points generation and thining

In [4]:
stable_ag_points = stable_ag_final_mask.selfMask().stratifiedSample(
    numPoints=100000,     
    classBand='ag', 
    region=config.ROI, 
    scale=30, 
    geometries=True,
    dropNulls=True
)

In [5]:
def apply_spatial_thinning(points, distance_meters):
    """
    Removes spatial autocorrelation by ensuring no two points 
    are within the specified distance of each other.
    """
    # 1. Add a random value to each point
    points_with_random = points.randomColumn('random')

    # 2. Define a spatial filter for the 1,000-meter radius
    dist_filter = ee.Filter.withinDistance(
        distance=distance_meters,
        leftField='.geo',
        rightField='.geo',
        maxError=10
    )

    # 3. Create a join to find all neighbors within that distance
    join = ee.Join.saveAll(
        matchesKey='neighbors',
        measureKey='distance'
    )

    # 4. Apply the join to the FeatureCollection
    joined_points = join.apply(points_with_random, points_with_random, dist_filter)

    # 5. Function to evaluate each neighborhood
    def check_if_max(feature):
        # Get the list of all points within 1,000m (including itself)
        neighbors = ee.List(feature.get('neighbors'))
        
        # Extract the random values of all these neighbors
        neighbor_randoms = neighbors.map(lambda f: ee.Feature(f).get('random'))
        
        # Find the maximum random value in this cluster
        max_random = neighbor_randoms.reduce(ee.Reducer.max())
        
        # If THIS point's random value is the maximum, mark it to be kept
        is_max = ee.Number(feature.get('random')).eq(max_random)
        
        return feature.set('keep', is_max)

    # 6. Apply the evaluation and filter out the losers
    thinned_points = joined_points.map(check_if_max).filter(ee.Filter.eq('keep', 1))

    # 7. Clean up the temporary properties we added so your data stays clean
    def cleanup(f):
        return f.set('keep', None).set('neighbors', None).set('random', None)
        
    return thinned_points.map(cleanup)

In [6]:
stable_ag_points_filtered = apply_spatial_thinning(stable_ag_points, 2000)
# geemap.ee_export_vector_to_asset(
#     collection=stable_ag_points_filtered,
#     description = "stable_ag_points_filtered",
#     assetId = "projects/ee-joshisur231/assets/stable_ag_points_filtered",
# )

In [8]:
stable_ag_points_filtered.size()

In [9]:
# Map.addLayer(stable_ag_final_mask, {"palette":["white", "red"]}, "stable_ag")
# Map.addLayer(stable_ag_points_filtered, {}, "stable_ag_points_filtered")
# Map

In [ ]:
# geemap.ee_export_vector_to_drive(filtered_samples, description="potential_stable_ag", fileFormat="kml")

Exporting potential_stable_ag... Please check the Task Manager from the JavaScript Code Editor.


### Step 6: Phenology Check

In [ ]:
start_date = "2000-01-01"
end_date = "2022-12-31"

In [ ]:
l8_ndvi_col = ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI")\
    .filterBounds(config.ROI)\
    .filterDate(start_date, end_date)